In [ ]:
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, peak_widths


# Add workflow utils to path
WORKFLOW_DIR =  Path("../workflow").absolute()
sys.path.insert(0, str(WORKFLOW_DIR))

# Set data path
DATASET = "gencode.v47"
DATA_PATH = f"../results/{DATASET}"

print("✓ Imports successful")

✓ Imports successful


In [ ]:
!ls -la {DATA_PATH}

total 14398008
drwxr-xr-x   7 dgarcia users         341 May 22 15:17 .
drwxr-xr-x   9 dgarcia users         341 May 21 15:44 ..
drwxr-xr-x   2 dgarcia users       12426 May 20 12:56 batches
-rw-r--r--   1 dgarcia users           0 May 20 12:56 .batches_created
-rw-r--r--   1 dgarcia users 12030400879 May 21 11:14 detection.csv
drwxr-xr-x 388 dgarcia users        8878 May 20 20:12 detections
drwxr-xr-x   3 dgarcia users         186 May 22 16:52 g4hunter
drwxr-xr-x 388 dgarcia users        8878 May 20 20:12 predictions
-rw-r--r--   1 dgarcia users    48895458 May 21 11:06 rG4detector_prediction.csv
-rw-r--r--   1 dgarcia users    87910264 May 21 16:08 rg4_summary_annotated.csv
-rw-r--r--   1 dgarcia users    87910264 May 21 15:07 rg4_summary.csv
drwxr-xr-x   5 dgarcia users          90 May 21 15:44 viz
-rw-r--r--   1 dgarcia users           0 May 21 16:18 .viz_done


In [ ]:
!head -n 5 {DATA_PATH}/detection.csv

ENST00000832824.1|ENSG00000290825.2|-|-|DDX11L16-260|DDX11L16|1379|lncRNA|,C,G,C,A,G,A,G,A,C,G,G,G,T,A,G,A,A,C,C,T,C,A,G,T,A,A,T,C,C,G,A,A,A,A,G,C,C,G,G,G,A,T,C,G,A,C,C,G,C,C,C,C,T,T,G,C,T,T,G,C,A,G,C,C,G,G,G,C,A,C,T,A,C,A,G,G,A,C,C,C,G,C,T,T,G,C,T,C,A,C,G,G,T,G,T,C,T,G,A,C,T,T,C,C,A,G,C,A,A,C,T,G,C,T,G,G,C,C,T,G,T,G,C,C,A,G,G,G,T,G,C,A,A,G,C,T,G,A,G,C,A,C,T,G,G,A,G,T,G,G,A,G,T,T,T,T,C,C,T,G,T,G,G,A,G,A,G,G,A,G,C,C,A,T,G,C,C,T,A,G,A,G,T,G,G,G,A,T,G,G,G,C,C,A,T,T,G,T,T,C,A,T,C,T,T,C,T,G,G,C,C,C,C,T,G,T,T,G,T,C,T,G,C,A,T,G,T,A,A,C,T,T,A,A,T,A,C,C,A,C,A,A,C,C,A,G,G,C,A,T,A,G,G,G,G,A,A,A,G,A,T,T,G,G,A,G,G,A,A,A,G,A,T,G,A,G,T,G,A,G,A,G,C,A,T,C,A,A,C,T,T,C,T,C,T,C,A,C,A,A,C,C,T,A,G,G,C,C,A,G,T,G,T,G,T,G,G,T,G,A,T,G,C,C,A,G,G,C,A,T,G,C,C,C,T,T,C,C,C,C,A,G,C,A,T,C,A,G,G,T,C,T,C,C,A,G,A,G,C,T,G,C,A,G,A,A,G,A,C,G,A,C,G,G,C,C,G,A,C,T,T,G,G,A,T,C,A,C,A,C,T,C,T,T,G,T,G,A,G,T,G,T,C,C,C,C,A,G,T,G,T,T,G,C,A,G,A,G,T,T,T,C,A,C,C,A,G,A,A,G,T,A,G,G,C,C,T,C,T,T,C,C,T,G,A,C,A,G,G,C,A,G,C,T,G,C,A,C,C,A,C,T,G

In [ ]:
DET_PATH = Path(DATA_PATH) / "detection.csv"
# This file has a particular format, where each transcripts has two rows of data
with open(DET_PATH, "r") as f:
    while True:
        line = f.readline()
        if line.startswith("ENST00000164024.5"):
            header, data1 = line.strip().split(",", maxsplit=1)
            data2 = f.readline().strip()
            break
        else:
            continue


## How to find G4-forming sequences?

I can think of two ways:
1. Find center of peaks and extrapolate coordinates
    - Use `find_peaks()` to find local maxima of RSR-ratio values with position `i` above a threshold `th`
    - Extract sequence `seq[i-30:i+1]`

2. Find width of peaks and extrapolate coordinates
    - Use `find_peaks()` as above.
    - Find edge coordinates (e.g. either manually or using `peak_widths()`)
    - Extract sequence `seq[left_edge:right_edge+1]`

In [ ]:
seq = "".join(data1.split(","))

# data2 contains all 
data_list = [float(x) for x in data2.strip(' ,').split(",")]

threshold = 1.56

peaks = find_peaks(data_list, height=threshold)
peak_centers = peaks[0]

### Approach 1

In [ ]:
for c in peak_centers:
    print(f"Peak at {c} with height {data_list[c]}")
    print(f"\tSequence: {format_g4_str(seq[c-30:c+1])}")

Peak at 145 with height 2.0795471312917106
	Sequence: cGGGGGtgcggagagacaggcggaGGGGcgg
Peak at 476 with height 1.8346926447229275
	Sequence: ggagtcttccGGGGtccGGGaggatGGGGGG
Peak at 10489 with height 2.4958850912651753
	Sequence: ctgtctcctGGGGGGtttgaggctGGGctgg
Peak at 11604 with height 1.6122549201141958
	Sequence: taaccacGGGGGagagtggctgtgcaGGGct


In [ ]:
# Now let's check how many of these are actually G4 motifs
g4_motif = re.compile(r"G{3,}.+G{3,}.+G{3,}.+G{3,}")

for c in peak_centers:
    has_g4_motif = g4_motif.search(seq[c-30:c+1])
    print(f"Peak at {c} with height {data_list[c]:.2f}")
    print(f"\tSequence: {format_g4_str(seq[c-30:c+1])} {'G4' if has_g4_motif else ' '}")


Peak at 145 with height 2.08
	Sequence: cGGGGGtgcggagagacaggcggaGGGGcgg  
Peak at 476 with height 1.83
	Sequence: ggagtcttccGGGGtccGGGaggatGGGGGG  
Peak at 10489 with height 2.50
	Sequence: ctgtctcctGGGGGGtttgaggctGGGctgg  
Peak at 11604 with height 1.61
	Sequence: taaccacGGGGGagagtggctgtgcaGGGct  


### Approach 2

The most simple G4 should look like `GGGNGGGNGGGNGGG`, so all G4s should be at least 16 nt long.
We can filter peaks by their width above the threshold

In [ ]:
signal = np.array(data_list)
above = signal > threshold
# Label connected regions
from scipy.ndimage import label
labeled, num_features = label(above)

In [ ]:
for region_id in range(1, num_features + 1):
    indices = np.where(labeled == region_id)[0]
    left_edge = indices[0]
    right_edge = indices[-1]
    center = indices[np.argmax(signal[indices])]
    width = right_edge - left_edge
    if width >= 16:
        print(f"Peak at {center}:\tedges [{left_edge}, {right_edge}]\twidth {width}\theight {signal[center]}")

Peak at 145:	edges [134, 169]	width 35	height 2.0795471312917106
Peak at 476:	edges [468, 484]	width 16	height 1.8346926447229275
Peak at 10489:	edges [10476, 10501]	width 25	height 2.4958850912651753


In [ ]:
def format_g4_str(seq):
    """
    Formats a G4-forming sequence by leaving G-tracts intact and lowering all other characters.
    G-tracts are runs of 3 or more consecutive G's
    """
    # Use regex to find G-tracts
    import re
    g4_str = ""
    last_end = 0
    for match in re.finditer(r"G{3,}", seq):
        start, end = match.span()
        # Add non-G characters in lowercase
        g4_str += seq[last_end:start].lower()
        # Add G-tract in uppercase
        g4_str += seq[start:end]
        last_end = end
    # Add any remaining non-G characters in lowercase
    g4_str += seq[last_end:].lower()
    return g4_str



In [ ]:
for region_id in range(1, num_features + 1):
    indices = np.where(labeled == region_id)[0]
    left_edge = indices[0]
    right_edge = indices[-1]
    has_g4_motif = g4_motif.search(seq[left_edge:right_edge+1])
    center = indices[np.argmax(signal[indices])]
    width = right_edge - left_edge
    if width >= 16:
        print(f"Peak at {center}:\tedges [{left_edge}, {right_edge}]\twidth {width}\theight {signal[center]} {'G4' if has_g4_motif else ' '}")
        print(f"\tSequence: {format_g4_str(seq[left_edge:right_edge+1])}")

Peak at 145:	edges [134, 169]	width 35	height 2.0795471312917106 G4
	Sequence: gcggaGGGGcGGGGGcccGGGGcggcggcaGGGGcc
Peak at 476:	edges [468, 484]	width 16	height 1.8346926447229275  
	Sequence: gatGGGGGGcctggcct
Peak at 10489:	edges [10476, 10501]	width 25	height 2.4958850912651753 G4
	Sequence: tgaggctGGGctGGGaGGGGGaGGGa


In [ ]:
for region_id in range(1, num_features + 1):
    indices = np.where(labeled == region_id)[0]
    left_edge = indices[0] - 20
    right_edge = indices[-1] + 20
    has_g4_motif = g4_motif.search(seq[left_edge:right_edge+1])
    center = indices[np.argmax(signal[indices])]
    width = right_edge - left_edge
    if width >= 16 + 40:
        print(f"Peak at {center}:\tedges [{left_edge}, {right_edge}]\twidth {width}\theight {signal[center]} {'G4' if has_g4_motif else ' '}")
        print(f"\tSequence: {format_g4_str(seq[left_edge:right_edge+1])}")
    else:
        print(f"Peak at {center}:\tedges [{left_edge}, {right_edge}]\twidth {width}\theight {signal[center]} {'g4' if has_g4_motif else 'X'}")
        print(f"\tSequence: {format_g4_str(seq[left_edge:right_edge+1])}")

Peak at 145:	edges [114, 189]	width 75	height 2.0795471312917106 G4
	Sequence: gcGGGGGtgcggagagacaggcggaGGGGcGGGGGcccGGGGcggcggcaGGGGcccGGGaGGGGGcccgagcggc
Peak at 476:	edges [448, 504]	width 56	height 1.8346926447229275 G4
	Sequence: agtcttccGGGGtccGGGaggatGGGGGGcctggcctGGGGGtcaGGGagcctatct
Peak at 10489:	edges [10456, 10521]	width 65	height 2.4958850912651753 G4
	Sequence: tttctgtctcctGGGGGGtttgaggctGGGctGGGaGGGGGaGGGagatagaGGGagagatgcagt
Peak at 11604:	edges [11581, 11626]	width 45	height 1.6122549201141958 X
	Sequence: GGGGGagagtggctgtgcaGGGctGGGGGGtggtctgtgcagacac


In [ ]:
import utils.rg4_parsing as rg4_parsing

DET_PATH = Path(DATA_PATH) / "detection.csv"
th = 1.56

# Initialize counters
tx_ids = []
tx_count = 0
peak_counts = []
for transcript_id, scores, positions in rg4_parsing.stream_detection(DET_PATH):
    tx_count += 1
    if tx_count % 10_000 == 0:
        print(f"Processed {tx_count} transcripts...")
    # Process each transcript
    #print(f"Processing transcript: {transcript_id}")
    # Find peaks > th
    scores_array = np.array(scores)
    peaks, _ = find_peaks(scores_array, height=th)
    # Count peaks
    peak_count = len(peaks)
    tx_ids.append(transcript_id)
    peak_counts.append(peak_count)

Processed 10000 transcripts...
Processed 20000 transcripts...
Processed 30000 transcripts...
Processed 40000 transcripts...
Processed 50000 transcripts...
Processed 60000 transcripts...
Processed 70000 transcripts...
Processed 80000 transcripts...
Processed 90000 transcripts...
Processed 100000 transcripts...
Processed 110000 transcripts...
Processed 120000 transcripts...
Processed 130000 transcripts...
Processed 140000 transcripts...
Processed 150000 transcripts...
Processed 160000 transcripts...
Processed 170000 transcripts...
Processed 180000 transcripts...
Processed 190000 transcripts...
Processed 200000 transcripts...
Processed 210000 transcripts...
Processed 220000 transcripts...
Processed 230000 transcripts...
Processed 240000 transcripts...
Processed 250000 transcripts...
Processed 260000 transcripts...
Processed 270000 transcripts...
Processed 280000 transcripts...
Processed 290000 transcripts...
Processed 300000 transcripts...
Processed 310000 transcripts...
Processed 320000 

In [ ]:
DET_PATH = Path(DATA_PATH) / "detection.csv"
th = 1.56

# Initialize counters
tx_ids = []
tx_count = 0
peak_counts = []
tx_lengths = []
for transcript_id, scores, positions in rg4_parsing.stream_detection(DET_PATH):
    tx_count += 1
    if tx_count % 10_000 == 0:
        print(f"Processed {tx_count} transcripts...")
    # Process each transcript
    #print(f"Processing transcript: {transcript_id}")
    # Find peaks > th
    scores_array = np.array(scores)
    peaks, _ = find_peaks(scores_array, height=th)
    # Count peaks
    peak_count = len(peaks)
    tx_ids.append(transcript_id)
    tx_lengths.append(len(scores))
    peak_counts.append(peak_count)

Processed 10000 transcripts...
Processed 20000 transcripts...
Processed 30000 transcripts...
Processed 40000 transcripts...
Processed 50000 transcripts...
Processed 60000 transcripts...
Processed 70000 transcripts...
Processed 80000 transcripts...
Processed 90000 transcripts...
Processed 100000 transcripts...
Processed 110000 transcripts...
Processed 120000 transcripts...
Processed 130000 transcripts...
Processed 140000 transcripts...
Processed 150000 transcripts...
Processed 160000 transcripts...
Processed 170000 transcripts...
Processed 180000 transcripts...
Processed 190000 transcripts...
Processed 200000 transcripts...
Processed 210000 transcripts...
Processed 220000 transcripts...
Processed 230000 transcripts...
Processed 240000 transcripts...
Processed 250000 transcripts...
Processed 260000 transcripts...
Processed 270000 transcripts...
Processed 280000 transcripts...
Processed 290000 transcripts...
Processed 300000 transcripts...
Processed 310000 transcripts...
Processed 320000 

In [ ]:
# Create DataFrame
df = pd.DataFrame({
    "transcript_id": tx_ids,
    "peak_count": peak_counts
})
df.sort_values("peak_count", ascending=False).head(20)

,transcript_id,peak_count
380420,ENST00000674361.1|ENSG00000241743.5|OTTHUMG000...,27
374678,ENST00000645032.1|ENSG00000156313.15|OTTHUMG00...,18
214217,ENST00000597346.1|ENSG00000269821.2|OTTHUMG000...,15
232988,ENST00000533871.8|ENSG00000080854.16|OTTHUMG00...,14
144651,ENST00000222990.8|ENSG00000106266.11|OTTHUMG00...,13
25391,ENST00000483825.6|ENSG00000198821.12|OTTHUMG00...,12
346221,ENST00000334976.11|ENSG00000161671.17|OTTHUMG0...,12
242875,ENST00000267116.8|ENSG00000139645.11|OTTHUMG00...,11
369750,ENST00000402918.7|ENSG00000166897.16|OTTHUMG00...,11
247847,ENST00000626826.1|ENSG00000281344.1|OTTHUMG000...,11


In [ ]:
binary_file = "/mnt/cbib/LNClassifier/paper/results/gencode.v47.common.cdhit.cv/tables/gencode.v47.common.cdhit.cv_binary_class_table.tsv"

binary_df = pd.read_csv(binary_file, sep="\t")
binary_df = binary_df[["seq_ID", "real"]]
binary_df.head()

,seq_ID,real
0,ENST00000000412.8,True
1,ENST00000002596.6,True
2,ENST00000002829.8,True
3,ENST00000005260.9,True
4,ENST00000005995.8,True


In [ ]:
df.index = df["transcript_id"].str.split("|").str[0]

df = df.merge(binary_df.set_index("seq_ID"), left_index=True, right_index=True)
df.head()


,transcript_id,peak_count,real_x,real_y,real
transcript_id,,,,,
ENST00000456328.3,ENST00000456328.3|ENSG00000290825.2|-|OTTHUMT0...,0,False,False,False
ENST00000417324.2,ENST00000417324.2|ENSG00000237613.3|OTTHUMG000...,0,False,False,False
ENST00000466430.5,ENST00000466430.5|ENSG00000241860.8|OTTHUMG000...,0,False,False,False
ENST00000484859.1,ENST00000484859.1|ENSG00000241860.8|OTTHUMG000...,1,False,False,False
ENST00000491962.2,ENST00000491962.2|ENSG00000241860.8|OTTHUMG000...,0,False,False,False


In [ ]:
from scipy.stats import chi2_contingency, mannwhitneyu

df["real"] = df["real"].astype(bool)
df["has_peak"] = df["peak_count"] > 0

# Chi-square test: has_peak vs class label
contingency = pd.crosstab(df["has_peak"], df["real"]).reindex(
    index=[False, True], columns=[False, True], fill_value=0
)
chi2, p_chi2, dof, expected = chi2_contingency(contingency)

# Effect size: Cramer's V
n = contingency.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * min(contingency.shape[0] - 1, contingency.shape[1] - 1)))

# Effect size: Odds Ratio (has_peak=True vs real=True)
a = contingency.loc[True, True]    # has_peak=True, real=True
b = contingency.loc[True, False]   # has_peak=True, real=False
c = contingency.loc[False, True]   # has_peak=False, real=True
d = contingency.loc[False, False]  # has_peak=False, real=False

if min(a, b, c, d) == 0:
    odds_ratio = ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))
    or_note = " (Haldane-Anscombe corrected)"
else:
    odds_ratio = (a * d) / (b * c)
    or_note = ""

print("Chi-square test (has_peak vs real)")
print(contingency)
print(f"chi2={chi2:.4f}, dof={dof}, p-value={p_chi2:.3e}")
print(f"Cramer's V={cramers_v:.4f}")
print(f"Odds Ratio={odds_ratio:.4f}{or_note}")

# Mann-Whitney U test: peak_count by class label
peak_true = df.loc[df["real"], "peak_count"]
peak_false = df.loc[~df["real"], "peak_count"]
u_stat, p_mwu = mannwhitneyu(peak_true, peak_false, alternative="two-sided")

# Effect size: Vargha-Delaney A (VDA)
vda = u_stat / (len(peak_true) * len(peak_false))

print("\nMann-Whitney U test (peak_count by real)")
print(f"n_true={len(peak_true)}, n_false={len(peak_false)}")
print(f"U={u_stat:.4f}, p-value={p_mwu:.3e}")
print(f"VDA (A12)={vda:.4f}")

Chi-square test (has_peak vs real)
real      False  True 
has_peak              
False     41428  56976
True       3848   9400
chi2=824.8103, dof=1, p-value=2.177e-181
Cramer's V=0.0859
Odds Ratio=1.7762

Mann-Whitney U test (peak_count by real)
n_true=66376, n_false=45276
U=1589073157.5000, p-value=8.894e-187
VDA (A12)=0.5288
